<a href="https://colab.research.google.com/github/dkalchenko01/UA-Locations-NER-Deep-Learning/blob/baseline/notebooks/04_bilstm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 04 — BiLSTM + BPE + BIO baseline

**Pipeline:** original text → committed ByteLevel BPE → random trainable embeddings (128) → one bidirectional GRU (128 per direction) → dropout (0.2) → linear classifier (256 → 3) → BIO → original character spans → location strings → `comp_metric`.

Use the fixed validation manifest, all UK training rows, and its existing 100,000-row RU subset. No cleaning, resplitting, vocabulary fitting, pretrained embeddings, CRF, attention, or augmentation. Five epochs; no early stopping. Best model: UK competition F1.

This notebook has been prepared by static review only. **No runtime results are supplied.** Assignment/course PDFs were unavailable; no claims about their contents are made. `uk_testlike` is derived from validation and is not an independent holdout.



## 1. Environment and repository

Select a GPU runtime in Kaggle or Colab. In Kaggle, attach `kse-ua-location-extraction-2026`, enable Internet for setup, and configure the `GITHUB_TOKEN` secret if a clone is needed. In Colab, configure the same secret and authenticate Kaggle through its supported credentials after accepting the competition rules. Do not put credentials in notebook cells or outputs.

An existing clone may be supplied using `UA_NER_REPO_DIR`. Otherwise the cell clones the private repository's **baseline** branch into the environment's working directory. An existing directory must already contain this implementation; it is never reset or pulled automatically. No datasets are stored in Git. Execute cells in order, including dependency installation, before imports.



In [1]:
import os
import sys
import subprocess
from pathlib import Path

ENVIRONMENT = "colab" if "google.colab" in sys.modules else "kaggle"
WORK_DIR = Path("/content" if ENVIRONMENT == "colab" else "/kaggle/working")
if not WORK_DIR.is_dir():
    raise RuntimeError("Run in Kaggle or Colab, or adapt the environment setup explicitly")
REPO_DIR = Path(os.environ.get("UA_NER_REPO_DIR", WORK_DIR / "UA-Locations-NER-Deep-Learning"))
if not REPO_DIR.exists():
    import base64
    if ENVIRONMENT == "colab":
        from google.colab import userdata
        github_token = userdata.get("GITHUB_TOKEN")
    else:
        from kaggle_secrets import UserSecretsClient
        github_token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    if not github_token:
        raise RuntimeError("Provide GITHUB_TOKEN as an environment secret or supply an existing clone")
    auth = base64.b64encode(f"x-access-token:{github_token}".encode()).decode()
    clone_env = os.environ.copy()
    clone_env.update(GIT_CONFIG_COUNT="1", GIT_CONFIG_KEY_0="http.extraheader",
                     GIT_CONFIG_VALUE_0=f"AUTHORIZATION: basic {auth}")
    try:
        subprocess.run(["git", "clone", "--branch", "baseline",
                        "https://github.com/dkalchenko01/UA-Locations-NER-Deep-Learning.git",
                        str(REPO_DIR)], env=clone_env, check=True)
    finally:
        del github_token, auth, clone_env
assert (REPO_DIR / "src/model.py").is_file(), "Supply a clone containing the baseline implementation"
sys.path.insert(0, str(REPO_DIR))
print("Environment:", ENVIRONMENT, "Repository:", REPO_DIR)

Environment: colab Repository: /content/UA-Locations-NER-Deep-Learning


In [2]:
%pip install -q -r "$REPO_DIR/requirements.txt"

In [3]:
import gc
import hashlib
import json
import platform
import random
import uuid
from datetime import datetime, timezone
from functools import partial
from importlib.metadata import version

import numpy as np
import pandas as pd
import torch
from tokenizers import Tokenizer
from torch import nn
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

from src.data_io import get_data_dir, load_manifest, load_data_splits
from src.eda_scan import parse_spans, has_pair_shape
from src.preprocessing import prepare_bio_data, bio_to_spans
from src.dataset import (NERDataset, LengthBatchSampler, collate_ner,
                         LABEL_TO_ID, ID_TO_LABEL, IGNORE_INDEX)
from src.metrics import comp_metric, location_errors


## 2. Configuration, reproducibility and output isolation

Full sequences are retained (`max_sequence_length=None`). The 16,384 padded-token budget changes batching only; a sequence exceeding that budget runs alone. This cannot guarantee GPU memory for an arbitrarily long single sequence. If memory is insufficient, lower batch size/token budget and restart the complete run; do not silently truncate or change the split.

The seed, actual dependency versions, source revision, tokenizer/manifest hashes and device are recorded. Deterministic PyTorch operations are requested; an unsupported operation raises rather than silently becoming nondeterministic. Exact reproducibility across different hardware/library versions is not guaranteed.



In [4]:
CONFIG = {
    "seed": 17, "embedding_dim": 128, "hidden_size": 128, "dropout": 0.2,
    "num_layers": 1, "bidirectional": True, "batch_size": 128,
    "learning_rate": 1e-3, "optimizer": "Adam", "epochs": 5,
    "gradient_clip": 1.0, "max_sequence_length": None,
    "max_padded_tokens": 16384, "pool_size": 4096, "prepare_batch_size": 2000,
    "num_workers": 0, "sources": ["uk", "ru"],
    "tokenizer": "bpe_bytelevel.json", "selection_metric": "uk_comp_f1",
    "early_stopping": False, "label_to_id": LABEL_TO_ID,
    "ignore_index": IGNORE_INDEX, "environment": ENVIRONMENT, "model": "bilstm",
}
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
random.seed(CONFIG["seed"])
np.random.seed(CONFIG["seed"])
torch.manual_seed(CONFIG["seed"])
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CONFIG["seed"])
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.use_deterministic_algorithms(True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
RUN_ID = "bilstm_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8]
RUN_DIR = WORK_DIR / "outputs" / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=False)

def save_json(path, value):
    with Path(path).open("w", encoding="utf-8") as handle:
        json.dump(value, handle, ensure_ascii=False, indent=2, allow_nan=False)

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

SPLITS_DIR = REPO_DIR / "data/splits"
TOKENIZER_PATH = REPO_DIR / "artifacts/tokenizers" / CONFIG["tokenizer"]
revision = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
worktree_status = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "status", "--short"], text=True).strip()
PROVENANCE = {
    "run_id": RUN_ID, "seed": CONFIG["seed"], "git_revision": revision,
    "git_status": worktree_status, "python": platform.python_version(),
    "packages": {name: version(name) for name in ["numpy", "pandas", "pyarrow", "torch", "tokenizers", "tqdm"]},
    "device": str(DEVICE), "cuda_version": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else None,
    "tokenizer_sha256": sha256_file(TOKENIZER_PATH),
    "manifest_sha256": sha256_file(SPLITS_DIR / "validation_manifest.parquet"),
    "implementation_sha256": {
      str(path.relative_to(REPO_DIR)): sha256_file(path)
      for path in [
          *sorted((REPO_DIR / "src").glob("*.py")),
          REPO_DIR / "requirements.txt",
      ]
    },
    "diagnostic_sha256": {name: sha256_file(SPLITS_DIR / f"{name}.parquet")
                          for name in ["uk_testlike", "uk_testlike_original"]},
}
save_json(RUN_DIR / "config.json", CONFIG)
save_json(RUN_DIR / "provenance.json", PROVENANCE)
print("Device:", DEVICE, "Run:", RUN_DIR)
print("CPU execution of the full experiment can be slow.")



Device: cuda Run: /content/outputs/bilstm_20261110T192416Z_a87f4fa7
CPU execution of the full experiment can be slow.


## 3. Fixed manifest and existing tokenizer

`load_data_splits` calls `get_data_dir`, `load_manifest` and `select_manifest_rows`. It returns DataFrames in RAM, not a streaming training dataset. Load one source at a time, retain only required columns, prepare in small batches and release each DataFrame. Token arrays use compact NumPy storage; training records discard character offsets after alignment. Validation retains original text and offsets for complete competition evaluation.

No original `is_valid` flags replace the committed manifest. UK grouping is by near-duplicate group; RU grouping is by `doc_id`. Group IDs are scoped by source.



In [5]:
manifest = load_manifest(SPLITS_DIR)
assert not manifest.duplicated(["source", "row_id"]).any()
assert set(manifest["source"]) == {"uk", "ru"}
assert set(manifest["split"]) == {"train", "val"}
assert set(manifest["seed"]) == {CONFIG["seed"]}
assert not manifest["group_id"].isna().any()
for source in CONFIG["sources"]:
    part = manifest[manifest["source"] == source]
    train_part, val_part = part[part["split"] == "train"], part[part["split"] == "val"]
    assert set(train_part["row_id"]).isdisjoint(val_part["row_id"])
    assert set(train_part["group_id"]).isdisjoint(val_part["group_id"])
counts = pd.crosstab(manifest["source"], manifest["split"])
assert counts.loc["uk", "train"] == 805983 and counts.loc["uk", "val"] == 201498
assert counts.loc["ru", "train"] == 100000 and counts.loc["ru", "val"] == 49369
display(counts)
counts.to_csv(RUN_DIR / "manifest_counts.csv")

tokenizer = Tokenizer.from_file(str(TOKENIZER_PATH))
tokenizer.no_padding()
tokenizer.no_truncation()
PAD_ID = tokenizer.token_to_id("[PAD]")
assert PAD_ID == 0 and tokenizer.token_to_id("[UNK]") == 1
assert tokenizer.get_vocab_size() == 32000
DATA_DIR = get_data_dir(ENVIRONMENT)
print("Data directory:", DATA_DIR)
print("Tokenizer vocabulary:", tokenizer.get_vocab_size())



split,train,val
source,,
ru,100000,49369
uk,805983,201498


100%|██████████| 811M/811M [00:04<00:00, 188MB/s]

Extracting files...


Data directory: /root/.cache/kagglehub/competitions/kse-ua-location-extraction-2026
Tokenizer vocabulary: 32000


## 4. BIO preparation and coverage audit

Reuse `prepare_bio_data` without changing its exact span-boundary policy. Each preparation batch is validated structurally first. Unexpected malformed/overlapping gold annotations stop the run. Alignment exclusions are logged by original source and row ID, never converted to fabricated O labels.

Training excludes unaligned or zero-token examples. **Every original validation row remains in inference/scoring**, including alignment failures; only its supervised loss is unavailable. Gold competition strings always come from original annotations. Empty texts receive a masked dummy token for packing and produce no spans. ByteLevel may give repeated/overlapping character offsets for Unicode bytes; these are preserved.

The saved preprocessing notebook reports ByteLevel training skips UK=1,602/RU=110 and validation skips UK=361/RU=44. These are reference diagnostics, not results from this run; actual counts are measured below. Round-trip checks apply to every aligned example.



In [6]:
ALIGNMENT_STATS = []
AUDIT_PATH = RUN_DIR / "alignment_exclusions.jsonl"

def checked_spans(text, raw):
    spans = parse_spans(raw)
    if not isinstance(text, str) or not has_pair_shape(spans):
        raise ValueError("Malformed text/annotation: inspect original source row")
    if not all(0 <= start < end <= len(text) for start, end in spans):
        raise ValueError("Out-of-bounds gold span")
    ordered = sorted(spans)
    if any(right[0] < left[1] for left, right in zip(ordered, ordered[1:])):
        raise ValueError("Overlapping gold spans cannot be represented by this BIO baseline")
    return spans


def prepare_frame(df, source, split, keep_for_evaluation):
    records = []
    stats = {"source": source, "split": split, "total": len(df),
             "aligned": 0, "alignment_failures": 0, "empty_token_sequences": 0,
             "supervised": 0, "retained": 0}
    with AUDIT_PATH.open("a", encoding="utf-8") as audit:
        for start in tqdm(range(0, len(df), CONFIG["prepare_batch_size"]),
                          desc=f"Prepare {source}/{split}"):
            batch = df.iloc[start:start + CONFIG["prepare_batch_size"]]
            gold_by_id = {}
            for row_id, row in batch.iterrows():
                gold_by_id[f"{source}_{row_id}"] = checked_spans(row["text"], row["loc_markers"])
            prepared, skipped = prepare_bio_data(tokenizer, batch, source)
            by_id = {record["text_id"]: record for record in prepared}
            skipped = set(skipped)
            assert len(by_id) + len(skipped) == len(batch)
            for row_id, row in batch.iterrows():
                text_id = f"{source}_{row_id}"
                original = by_id.get(text_id)
                spans = gold_by_id[text_id]
                if original is None:
                    assert text_id in skipped
                    stats["alignment_failures"] += 1
                    audit.write(json.dumps({"text_id": text_id, "source": source,
                                            "row_id": int(row_id), "split": split,
                                            "reason": "span_token_alignment"}) + "\n")
                    if not keep_for_evaluation:
                        continue
                    encoding = tokenizer.encode(row["text"], add_special_tokens=False)
                    ids, offsets, labels = encoding.ids, encoding.offsets, None
                else:
                    stats["aligned"] += 1
                    ids, offsets = original["input_ids"], original["offsets"]
                    assert sorted(bio_to_spans(original["labels"], offsets)) == sorted(spans)
                    labels = np.asarray([LABEL_TO_ID[value] for value in original["labels"]], dtype=np.int8)
                assert len(ids) == len(offsets)
                assert all(0 <= a <= b <= len(row["text"]) for a, b in offsets)
                if not ids:
                    stats["empty_token_sequences"] += 1
                    audit.write(json.dumps({"text_id": text_id, "source": source,
                                            "row_id": int(row_id), "split": split,
                                            "reason": "empty_token_sequence"}) + "\n")
                    if not keep_for_evaluation:
                        continue
                    labels = None
                if labels is not None:
                    stats["supervised"] += 1
                record = {"text_id": text_id, "source": source, "row_id": int(row_id),
                          "input_ids": np.asarray(ids, dtype=np.int32), "labels": labels}
                if keep_for_evaluation:
                    record.update(text=row["text"], offsets=np.asarray(offsets, dtype=np.int32).reshape(-1, 2),
                                  gold_spans=spans,
                                  gold_locations=[row["text"][a:b] for a, b in spans])
                records.append(record)
    stats["retained"] = len(records)
    if keep_for_evaluation:
        assert len(records) == len(df)
    ALIGNMENT_STATS.append(stats)
    return records

train_records, val_records = [], []
for source in CONFIG["sources"]:
    frames = load_data_splits(ENVIRONMENT, SPLITS_DIR, sources=(source,))
    for split in ["train", "val"]:
        frame = frames.pop(f"{source}_{split}")[["text", "loc_markers"]]
        expected = manifest[(manifest["source"] == source) & (manifest["split"] == split)]["row_id"]
        assert frame.index.tolist() == expected.tolist()
        prepared_records = prepare_frame(frame, source, split, keep_for_evaluation=(split == "val"))
        (train_records if split == "train" else val_records).extend(prepared_records)
        del frame, prepared_records
        gc.collect()
    del frames
assert len(val_records) == int(counts["val"].sum())
assert len({record["text_id"] for record in val_records}) == len(val_records)
alignment_table = pd.DataFrame(ALIGNMENT_STATS)
display(alignment_table)
alignment_table.to_csv(RUN_DIR / "alignment_counts.csv", index=False)
del manifest
gc.collect()



Prepare uk/train:   0%|          | 0/403 [00:00<?, ?it/s]

Prepare uk/val:   0%|          | 0/101 [00:00<?, ?it/s]

Prepare ru/train:   0%|          | 0/50 [00:00<?, ?it/s]

Prepare ru/val:   0%|          | 0/25 [00:00<?, ?it/s]

,source,split,total,aligned,alignment_failures,empty_token_sequences,supervised,retained
0,uk,train,805983,804381,1602,0,804381,804381
1,uk,val,201498,201137,361,0,201137,201498
2,ru,train,100000,99890,110,0,99890,99890
3,ru,val,49369,49325,44,0,49325,49369


0

## 5. Dataset and dynamic batches

Training uses shuffled pools sorted by length, then shuffles the resulting batches. Every retained record appears once per epoch. Validation batches can be reordered for efficiency: results are always keyed by original `text_id`. The collator creates padding masks and ignores unavailable/padded labels with `-100`. Packing prevents both GRU directions from learning from padding.



In [7]:
train_dataset, val_dataset = NERDataset(train_records), NERDataset(val_records)
train_sampler = LengthBatchSampler(train_dataset.lengths, CONFIG["batch_size"], shuffle=True,
                                  seed=CONFIG["seed"], max_padded_tokens=CONFIG["max_padded_tokens"],
                                  pool_size=CONFIG["pool_size"])
val_sampler = LengthBatchSampler(val_dataset.lengths, CONFIG["batch_size"],
                                max_padded_tokens=CONFIG["max_padded_tokens"])
loader_options = {"collate_fn": partial(collate_ner, pad_id=PAD_ID),
                  "num_workers": CONFIG["num_workers"], "pin_memory": DEVICE.type == "cuda"}
train_loader = DataLoader(train_dataset, batch_sampler=train_sampler, **loader_options)
val_loader = DataLoader(val_dataset, batch_sampler=val_sampler, **loader_options)
for name, dataset in [("train", train_dataset), ("val", val_dataset)]:
    lengths = np.asarray(dataset.lengths)
    print(name, "rows:", len(dataset), "length percentiles:", np.percentile(lengths, [50, 95, 99, 100]))
    print("Single sequences exceeding batching token budget:", int((lengths > CONFIG["max_padded_tokens"]).sum()))
    array_bytes = sum(value.nbytes for record in dataset.records for value in record.values()
                      if isinstance(value, np.ndarray))
    print("Compact array storage (GiB; excludes Python/text/loader overhead):", round(array_bytes / 1024**3, 3))



train rows: 904271 length percentiles: [ 21.  45.  57. 302.]
Single sequences exceeding batching token budget: 0
Compact array storage (GiB; excludes Python/text/loader overhead): 0.097
val rows: 250867 length percentiles: [ 21.  46.  60. 295.]
Single sequences exceeding batching token budget: 0
Compact array storage (GiB; excludes Python/text/loader overhead): 0.073


## 6. Model, loss and optimizer

The GRU has exactly **one stacked recurrent layer**, with two directions. Input `[B,T]` becomes embeddings `[B,T,128]`, packed GRU outputs `[B,T,256]`, and logits `[B,T,3]`. `O=0`, `B-LOC=1`, `I-LOC=2`; padding is not a fourth class. Random embeddings are trainable, with the padding row fixed by `padding_idx`.

Use unweighted token cross-entropy. Summed loss is divided by the number of supervised tokens for each update and for epoch reporting. The validation denominator excludes unavailable gold BIO labels, while competition F1 includes all original rows.



In [8]:
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence


class BiLSTMTagger(nn.Module):
    def __init__(self, vocab_size, pad_id=0, embedding_dim=128,
                 hidden_size=128, dropout=0.2, num_labels=3):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size, embedding_dim, padding_idx=pad_id
        )

        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_size,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )

        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(2 * hidden_size, num_labels)

    def forward(self, input_ids, lengths):
        embedded = self.embedding(input_ids)

        packed = pack_padded_sequence(
            embedded,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False
        )

        encoded, _ = self.lstm(packed)

        encoded, _ = pad_packed_sequence(
            encoded,
            batch_first=True,
            total_length=input_ids.size(1)
        )

        logits = self.classifier(self.dropout(encoded))

        return logits


model = BiLSTMTagger(
    tokenizer.get_vocab_size(),
    pad_id=PAD_ID,
    embedding_dim=CONFIG["embedding_dim"],
    hidden_size=CONFIG["hidden_size"],
    dropout=CONFIG["dropout"],
    num_labels=len(LABEL_TO_ID)
).to(DEVICE)

criterion = nn.CrossEntropyLoss(
    ignore_index=IGNORE_INDEX,
    reduction="sum"
)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=CONFIG["learning_rate"]
)

print(model)
print(
    "Trainable parameters:",
    sum(p.numel() for p in model.parameters() if p.requires_grad)
)

BiLSTMTagger(
  (embedding): Embedding(32000, 128, padding_idx=0)
  (lstm): LSTM(128, 128, batch_first=True, bidirectional=True)
  (dropout): Dropout(p=0.2, inplace=False)
  (classifier): Linear(in_features=256, out_features=3, bias=True)
)
Trainable parameters: 4360963


## 7. Training and complete validation evaluation

The official metric in `src/metrics.py` is copied with the supplied literal first regex (no f-string), empty-string set members, empty-set rules and eight-decimal rounding intact. It must not be replaced with a conventional punctuation cleaner or token F1.

Argmax labels are decoded using the existing `bio_to_spans`; an orphan `I-LOC` starts a span, matching that function's established behavior. Extract locations using original `[start,end)` positions. A zero-width decoded span contains no text and is omitted. All other predictions, including duplicate strings, are passed to the metric, which performs its own normalization and set conversion.



In [9]:
def train_epoch(epoch):
    model.train()
    train_sampler.set_epoch(epoch)
    total_loss, total_tokens = 0.0, 0
    progress = tqdm(train_loader, desc=f"Epoch {epoch} train")
    for batch in progress:
        ids = batch["input_ids"].to(DEVICE, non_blocking=True)
        targets = batch["labels"].to(DEVICE, non_blocking=True)
        supervised = int(targets.ne(IGNORE_INDEX).sum().item())
        if not supervised:
            continue
        optimizer.zero_grad(set_to_none=True)
        logits = model(ids, batch["lengths"])
        loss_sum = criterion(logits.reshape(-1, len(LABEL_TO_ID)), targets.reshape(-1))
        loss = loss_sum / supervised
        if not torch.isfinite(loss):
            raise RuntimeError("Non-finite training loss")
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), CONFIG["gradient_clip"], error_if_nonfinite=True)
        optimizer.step()
        total_loss += loss_sum.item()
        total_tokens += supervised
        progress.set_postfix(loss=total_loss / total_tokens)
    if not total_tokens:
        raise RuntimeError("No supervised training tokens")
    return total_loss / total_tokens


@torch.no_grad()
def predict_loader(loader, description="Validation"):
    model.eval()
    predictions = {}
    loss_sums = {source: 0.0 for source in CONFIG["sources"]}
    token_counts = {source: 0 for source in CONFIG["sources"]}
    for batch in tqdm(loader, desc=description):
        ids = batch["input_ids"].to(DEVICE, non_blocking=True)
        targets = batch["labels"].to(DEVICE, non_blocking=True)
        logits = model(ids, batch["lengths"])
        predicted_ids = logits.argmax(dim=-1).cpu().numpy()
        # Token losses remain separate so per-source means use correct denominators.
        losses = nn.functional.cross_entropy(logits.transpose(1, 2), targets,
                                             ignore_index=IGNORE_INDEX, reduction="none")
        row_losses = losses.sum(dim=1).cpu().tolist()
        row_tokens = targets.ne(IGNORE_INDEX).sum(dim=1).cpu().tolist()
        for index, record in enumerate(batch["records"]):
            source, text_id = record["source"], record["text_id"]
            if source in loss_sums:
                loss_sums[source] += row_losses[index]
                token_counts[source] += row_tokens[index]
            n = len(record["input_ids"])
            labels = [ID_TO_LABEL[int(value)] for value in predicted_ids[index, :n]]
            offsets = [tuple(map(int, pair)) for pair in record["offsets"]]
            spans = bio_to_spans(labels, offsets)
            assert all(0 <= a <= b <= len(record["text"]) for a, b in spans)
            spans = [(a, b) for a, b in spans if a < b]
            if text_id in predictions:
                raise ValueError(f"Duplicate prediction ID: {text_id}")
            predictions[text_id] = {"spans": spans, "locations": [record["text"][a:b] for a, b in spans]}
    assert len(predictions) == len(loader.dataset)
    losses = {source: loss_sums[source] / token_counts[source] if token_counts[source] else None
              for source in CONFIG["sources"]}
    total_tokens = sum(token_counts.values())
    losses["combined"] = sum(loss_sums.values()) / total_tokens if total_tokens else None
    return predictions, losses


def competition_scores(records, predictions):
    assert len(records) == len(predictions)
    scores = {}
    for source in [*CONFIG["sources"], "combined"]:
        selected = records if source == "combined" else [r for r in records if r["source"] == source]
        assert selected, f"No evaluation rows for {source}"
        scores[f"{source}_comp_f1"] = comp_metric(
            [r["gold_locations"] for r in selected],
            [predictions[r["text_id"]]["locations"] for r in selected])
    return scores


def save_predictions(path, records, predictions):
    # Key lookup makes output independent of sampler ordering.
    with Path(path).open("w", encoding="utf-8") as handle:
        for record in records:
            output = {"text_id": record["text_id"], "source": record["source"],
                      "row_id": record["row_id"], **predictions[record["text_id"]]}
            if "gold_locations" in record:
                output.update(gold_locations=record["gold_locations"],
                              **location_errors(record["gold_locations"], output["locations"]))
            handle.write(json.dumps(output, ensure_ascii=False) + "\n")



## 8. Five epochs and best checkpoint

No early stopping or scheduler. Validate after every epoch and retain the first checkpoint attaining the highest rounded UK competition F1. Each epoch writes ID-linked validation predictions and history. Atomic checkpoint replacement only updates this run's own `best.pt`.

The checkpoint includes optimizer and RNG states for provenance/recovery; this notebook starts a fresh run and does not automatically resume. Only load checkpoints created by this notebook from a trusted source.



In [10]:
history = []
best_score, best_epoch = -1.0, None
BEST_PATH = RUN_DIR / "best.pt"
for epoch in range(1, CONFIG["epochs"] + 1):
    train_loss = train_epoch(epoch)
    predictions, val_losses = predict_loader(val_loader, f"Epoch {epoch} validation")
    scores = competition_scores(val_records, predictions)
    row = {"epoch": epoch, "train_loss": train_loss,
           **{f"{source}_val_loss": value for source, value in val_losses.items()}, **scores}
    history.append(row)
    pd.DataFrame(history).to_csv(RUN_DIR / "history.csv", index=False)
    save_predictions(RUN_DIR / f"val_predictions_epoch_{epoch}.jsonl", val_records, predictions)
    if scores["uk_comp_f1"] > best_score:
        best_score, best_epoch = scores["uk_comp_f1"], epoch
        checkpoint = {
            "epoch": epoch, "selection_score": best_score, "config": CONFIG,
            "sampler_epoch": epoch,
            "provenance": PROVENANCE, "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(), "history": list(history),
            "python_rng_state": random.getstate(), "numpy_rng_state": np.random.get_state(),
            "torch_rng_state": torch.get_rng_state(),
            "cuda_rng_state": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
        }
        temporary_path = RUN_DIR / "best.tmp.pt"
        torch.save(checkpoint, temporary_path)
        temporary_path.replace(BEST_PATH)
        del checkpoint
    print(row, "Best epoch:", best_epoch)
    del predictions
    gc.collect()
display(pd.DataFrame(history))
# Free optimizer states and training records before final inference.
optimizer.zero_grad(set_to_none=True)
del optimizer, train_loader, train_sampler, train_dataset, train_records
gc.collect()
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()



Epoch 1 train:   0%|          | 0/7110 [00:00<?, ?it/s]

Epoch 1 validation:   0%|          | 0/1973 [00:00<?, ?it/s]

{'epoch': 1, 'train_loss': 0.03983837786011864, 'uk_val_loss': 0.02170071597456532, 'ru_val_loss': 0.02943418591495171, 'combined_val_loss': 0.023638763173588376, 'uk_comp_f1': 0.83166495, 'ru_comp_f1': 0.80210623, 'combined_comp_f1': 0.82423818} Best epoch: 1


Epoch 2 train:   0%|          | 0/7119 [00:01<?, ?it/s]

Epoch 2 validation:   0%|          | 0/1973 [00:00<?, ?it/s]

{'epoch': 2, 'train_loss': 0.019368853728557685, 'uk_val_loss': 0.018251040472870895, 'ru_val_loss': 0.02382667757606331, 'combined_val_loss': 0.019648323704577773, 'uk_comp_f1': 0.85753433, 'ru_comp_f1': 0.8503805, 'combined_comp_f1': 0.85573024} Best epoch: 2


... (6 outputs left out)

Epoch 5 train:   0%|          | 0/7114 [00:00<?, ?it/s]

Epoch 5 validation:   0%|          | 0/1973 [00:00<?, ?it/s]

{'epoch': 5, 'train_loss': 0.009665945402829702, 'uk_val_loss': 0.019478103589369952, 'ru_val_loss': 0.024456788465595514, 'combined_val_loss': 0.020725787533778652, 'uk_comp_f1': 0.8634869, 'ru_comp_f1': 0.86559872, 'combined_comp_f1': 0.86402127} Best epoch: 4


,epoch,train_loss,uk_val_loss,ru_val_loss,combined_val_loss,uk_comp_f1,ru_comp_f1,combined_comp_f1
0,1,0.039838,0.021701,0.029434,0.023639,0.831665,0.802106,0.824238
1,2,0.019369,0.018251,0.023827,0.019648,0.857534,0.850380,0.855730
2,3,0.015122,0.017689,0.022351,0.018857,0.860882,0.869374,0.863026
3,4,0.012180,0.017962,0.022853,0.019188,0.866313,0.868048,0.866751
4,5,0.009666,0.019478,0.024457,0.020726,0.863487,0.865599,0.864021


## 9. Reload the selected model and inspect errors

Recompute validation predictions from the saved best weights. Diagnostics include normalized FP/FN, exact character-span entity precision/recall/F1, negative-text false positives, per-source scores and token-length bins. Exact-span metrics count occurrences; competition F1 counts normalized strings once per text. These answer different questions.



In [11]:
# weights_only=False is needed for this notebook's optimizer/NumPy RNG metadata.
# BEST_PATH was generated in this run; never substitute an untrusted checkpoint.
checkpoint = torch.load(BEST_PATH, map_location="cpu", weights_only=False)
assert checkpoint["config"] == CONFIG
assert checkpoint["provenance"]["tokenizer_sha256"] == PROVENANCE["tokenizer_sha256"]
model.load_state_dict(checkpoint["model_state_dict"])
selected_epoch = checkpoint["epoch"]
del checkpoint
best_predictions, best_losses = predict_loader(val_loader, "Best checkpoint validation")
best_scores = competition_scores(val_records, best_predictions)
assert best_scores["uk_comp_f1"] == best_score, "Reloaded model changed the selection score"
save_predictions(RUN_DIR / "val_predictions_best.jsonl", val_records, best_predictions)


def diagnostic_summary(records, predictions):
    tp = fp = fn = negative = negative_fp = 0
    for record in records:
        gold, guess = set(record["gold_spans"]), set(predictions[record["text_id"]]["spans"])
        tp += len(gold & guess)
        fp += len(guess - gold)
        fn += len(gold - guess)
        if not gold:
            negative += 1
            negative_fp += bool(guess)
    precision = tp / (tp + fp) if tp + fp else (1.0 if not fn else 0.0)
    recall = tp / (tp + fn) if tp + fn else 1.0
    return {"rows": len(records), "span_precision": precision, "span_recall": recall,
            "span_f1": 2 * precision * recall / (precision + recall) if precision + recall else 0.0,
            "no_location_rows": negative, "no_location_rows_with_fp": negative_fp,
            "no_location_fp_rate": negative_fp / negative if negative else None}


diagnostics = []
for source in [*CONFIG["sources"], "combined"]:
    selected = val_records if source == "combined" else [r for r in val_records if r["source"] == source]
    diagnostics.append({"source": source, **diagnostic_summary(selected, best_predictions)})
display(pd.DataFrame(diagnostics))
pd.DataFrame(diagnostics).to_csv(RUN_DIR / "validation_diagnostics.csv", index=False)
length_diagnostics = []
for lower, upper in [(0, 32), (33, 64), (65, 128), (129, None)]:
    selected = [r for r in val_records if len(r["input_ids"]) >= lower
                and (upper is None or len(r["input_ids"]) <= upper)]
    if selected:
        length_diagnostics.append({"min_tokens": lower, "max_tokens": upper, "rows": len(selected),
            "comp_f1": comp_metric([r["gold_locations"] for r in selected],
                                   [best_predictions[r["text_id"]]["locations"] for r in selected])})
display(pd.DataFrame(length_diagnostics))
pd.DataFrame(length_diagnostics).to_csv(RUN_DIR / "length_diagnostics.csv", index=False)
shown = 0
for record in val_records:
    guess = best_predictions[record["text_id"]]
    errors = location_errors(record["gold_locations"], guess["locations"])
    if errors["false_positives"] or errors["false_negatives"]:
        print(record["text_id"], record["text"])
        print("Gold:", record["gold_locations"], "Predicted:", guess["locations"], errors)
        shown += 1
        if shown == 10:
            break
save_json(RUN_DIR / "summary.json", {"best_epoch": selected_epoch, **best_scores,
                                    "validation_losses": best_losses, "diagnostics": diagnostics})
print("Selected epoch:", selected_epoch, best_scores)
del best_predictions



Best checkpoint validation:   0%|          | 0/1973 [00:00<?, ?it/s]

,source,rows,span_precision,span_recall,span_f1,no_location_rows,no_location_rows_with_fp,no_location_fp_rate
0,uk,201498,0.853769,0.872018,0.862797,154766,2736,0.017678
1,ru,49369,0.855950,0.877306,0.866496,33879,864,0.025503
2,combined,250867,0.854321,0.873352,0.863732,188645,3600,0.019083


,min_tokens,max_tokens,rows,comp_f1
0,0,32.0,200342,0.879247
1,33,64.0,48929,0.846696
2,65,128.0,1560,0.800370
3,129,NaN,36,0.791444


uk_79 Елі Кара та Уріель Кельман вилетіли до Гонконга одразу після звільнення з в'язниці "Маунт-Іден".
Gold: ['Гонконга одра'] Predicted: ['Гонконга'] {'false_positives': ['гонконга'], 'false_negatives': ['гонконга одра']}
uk_151 Вони бурхливо обговорювали це в загальному чаті у WhatsApp та писали, що не уявляють матчі проти "Барселони" без Мессі.
Gold: ['Мессі'] Predicted: [] {'false_positives': [], 'false_negatives': ['мессі']}
uk_609 У Росії чоловік стверджує, що iPhone "перетворив його на гея" - Світ - TCH.
Gold: ['Росії'] Predicted: ['Росії', 'Світ'] {'false_positives': ['світ'], 'false_negatives': []}
uk_887 Також у Тельців буде дуже приємна зустріч до кінця тижня, будьте готові.
Gold: ['Тельців'] Predicted: [] {'false_positives': [], 'false_negatives': ['тельців']}
uk_897 Ця книгa, нaдрукoвaнa в Oстрoзi рoсiйським eмiгрaнтoм Іваном Федорович Мoсквитинoм, з пoлiгрaфiчнoгo бoку є визнaним шeдeврoм слoв’янськoгo книгoдрукувaння, a тeкстoлoгiчнo i дoнинi ввaжaється кaнoнiчнoю.
Gold:

## 10. Validation-derived domain-shift diagnostic

Compare the saved 5,000 original UK validation sentences with their 1,000 concatenated/noisy posts. These are **not independent holdouts** and are never training data or checkpoint-selection data. The number of texts and per-text deduplication change, so their score difference reflects both context/style changes and competition set aggregation.

Parquet annotations are arrays rather than CSV strings. Serialize a temporary copy before calling the existing parser; do not edit the saved artifacts. The source-row membership and exact extracted gold strings are checked before inference.



In [13]:
original = pd.read_parquet(SPLITS_DIR / "uk_testlike_original.parquet")
testlike = pd.read_parquet(SPLITS_DIR / "uk_testlike.parquet")
assert len(original) == 5000 and len(testlike) == 1000
original_ids = set(map(int, original["row_id"]))
validation_uk_ids = {r["row_id"] for r in val_records if r["source"] == "uk"}
assert original_ids <= validation_uk_ids
used_ids = [int(value) for values in testlike["source_row_ids"] for value in values]
assert len(used_ids) == len(set(used_ids)) == len(original_ids)
assert set(used_ids) == original_ids


def serialize_parquet_spans(frame):
    result = frame[["text", "loc_markers"]].copy()

    def serialize(spans):
        if isinstance(spans, str):
            return spans
        return repr([tuple(map(int, span)) for span in spans])

    result["loc_markers"] = result["loc_markers"].map(serialize)
    return result


original_frame = serialize_parquet_spans(original)
original_frame.index = original["row_id"].to_numpy()
testlike_frame = serialize_parquet_spans(testlike)
original_diagnostic = prepare_frame(original_frame, "uk", "testlike_original", True)
testlike_diagnostic = prepare_frame(testlike_frame, "uk", "testlike", True)
original_gold = {r["row_id"]: r["gold_locations"] for r in original_diagnostic}
for record, source_ids in zip(testlike_diagnostic, testlike["source_row_ids"]):
    expected_gold = [location for row_id in source_ids for location in original_gold[int(row_id)]]
    assert record["gold_locations"] == expected_gold


def inference_loader(records):
    dataset = NERDataset(records)
    sampler = LengthBatchSampler(dataset.lengths, CONFIG["batch_size"],
                                 max_padded_tokens=CONFIG["max_padded_tokens"])
    return DataLoader(dataset, batch_sampler=sampler, **loader_options)


domain_scores = {}
for name, records in [("uk_testlike_original", original_diagnostic), ("uk_testlike", testlike_diagnostic)]:
    predictions, _ = predict_loader(inference_loader(records), name)
    domain_scores[name] = comp_metric([r["gold_locations"] for r in records],
                                     [predictions[r["text_id"]]["locations"] for r in records])
    save_predictions(RUN_DIR / f"{name}_predictions.jsonl", records, predictions)
    del predictions
save_json(RUN_DIR / "domain_shift_diagnostics.json", domain_scores)
pd.DataFrame(ALIGNMENT_STATS).to_csv(RUN_DIR / "alignment_counts.csv", index=False)
print("Validation-derived diagnostic scores:", domain_scores)
del original, testlike, original_frame, testlike_frame, original_diagnostic, testlike_diagnostic



Prepare uk/testlike_original:   0%|          | 0/3 [00:00<?, ?it/s]

Prepare uk/testlike:   0%|          | 0/1 [00:00<?, ?it/s]

uk_testlike_original:   0%|          | 0/40 [00:00<?, ?it/s]

uk_testlike:   0%|          | 0/9 [00:00<?, ?it/s]

Validation-derived diagnostic scores: {'uk_testlike_original': 0.85536455, 'uk_testlike': 0.76578521}


## 11. Kaggle test inference and submission

Use the selected checkpoint, full original text and the committed tokenizer. `test.csv.locations` is a placeholder, never gold. Predictions are keyed by `text_id` despite length-sorted inference, then joined back in original test order. Serialize location lists with `json.dumps(ensure_ascii=False)` and let pandas handle CSV quoting. No Kaggle submission is sent automatically.



In [14]:
test = pd.read_csv(DATA_DIR / "test.csv")
assert test["text_id"].is_unique and test["text_id"].notna().all()
assert test["text"].map(lambda value: isinstance(value, str)).all()
test_records = []
for row_id, row in test.iterrows():
    encoding = tokenizer.encode(row["text"], add_special_tokens=False)
    assert all(0 <= a <= b <= len(row["text"]) for a, b in encoding.offsets)
    test_records.append({"text_id": int(row["text_id"]), "row_id": int(row_id), "source": "test",
                         "text": row["text"], "input_ids": np.asarray(encoding.ids, dtype=np.int32),
                         "offsets": np.asarray(encoding.offsets, dtype=np.int32).reshape(-1, 2), "labels": None})
assert len({r["text_id"] for r in test_records}) == len(test)
test_predictions, _ = predict_loader(inference_loader(test_records), "Kaggle test")
assert set(test_predictions) == set(test["text_id"])
save_predictions(RUN_DIR / "test_predictions.jsonl", test_records, test_predictions)
submission = test[["text_id"]].copy()
submission["locations"] = submission["text_id"].map(
    lambda text_id: json.dumps(test_predictions[text_id]["locations"], ensure_ascii=False))
SUBMISSION_PATH = RUN_DIR / "submission.csv"
submission.to_csv(SUBMISSION_PATH, index=False)
roundtrip = pd.read_csv(SUBMISSION_PATH)
assert roundtrip.columns.tolist() == ["text_id", "locations"]
assert roundtrip["text_id"].tolist() == test["text_id"].tolist()
assert [json.loads(value) for value in roundtrip["locations"]] == [
    test_predictions[text_id]["locations"] for text_id in test["text_id"]]
display(submission.head())
print("Saved submission:", SUBMISSION_PATH)



Kaggle test:   0%|          | 0/6 [00:00<?, ?it/s]

,text_id,locations
0,0,[]
1,1,"[""Києві"", ""Шулявського шляхопроводу"", ""\n\n🤝"",..."
2,2,"[""Гог"", ""Миргородського району"", ""Полтавської ..."
3,3,[]
4,4,"[""проспекті Академіка Глушкова, 9"", ""\n""]"


Saved submission: /content/outputs/bilstm_20261110T192416Z_a87f4fa7/submission.csv


## 12. Results and export

Results are populated only by execution: `history.csv`, `summary.json`, `domain_shift_diagnostics.json`, ID-linked prediction JSONL files, alignment counts/exclusions, and `best.pt` belong to this unique run. No numerical model-performance claims are prefilled.

**First-run checks:** confirm manifest composition/group disjointness; tokenizer IDs/hash; alignment coverage and round trips; full validation row counts; finite losses; CPU/GPU memory; decreasing training loss; saved/reloaded best model; sensible original-text spans; and ID/JSON/CSV round trips for submission. Full runtime correctness and GPU capacity must be checked manually in your environment.

Kaggle: save a notebook version including outputs and download the run directory/archive from the Output panel. Colab: download the archive before the temporary runtime ends, using the Files panel or `files.download` below. Both the archive and checkpoint can be large. No Drive integration is required. Private-repository cloning works after this branch has been made available remotely; alternatively upload/provide a local clone containing these changes.



In [15]:
import shutil
archive_path = shutil.make_archive(str(WORK_DIR / RUN_ID), "zip", root_dir=RUN_DIR)
print("Export archive:", archive_path)
print("Run directory:", RUN_DIR)
# Colab: uncomment to trigger a browser download after inspecting the outputs.
# from google.colab import files
# files.download(archive_path)



Export archive: /content/bilstm_20261110T192416Z_a87f4fa7.zip
Run directory: /content/outputs/bilstm_20261110T192416Z_a87f4fa7


## Work done after original submission

In [16]:
import re
import string
from src.metrics import hard_processing as normalize_original


def normalize_fixed(text):
    text = re.sub(rf"[{re.escape(string.punctuation)}]", "", text)
    text = re.sub(r'\d', '', text)
    text = re.sub(r'\b\w\b\s?', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text.lower()


def evaluate_saved(path, normalize, source=None):
    tp = fp = fn = p = 0

    with open(path, encoding="utf-8") as file:
        for line in file:
            row = json.loads(line)

            if source is not None and row["source"] != source:
                continue

            true = {normalize(s) for s in row["gold_locations"]}
            pred = {normalize(s) for s in row["locations"]}

            tp += len(true & pred)
            fp += len(pred - true)
            fn += len(true - pred)
            p += len(true)

    precision = tp / (tp + fp) if tp + fp else (1.0 if p == 0 else 0.0)
    recall = tp / (tp + fn) if tp + fn else (1.0 if p == 0 else 0.0)

    return round(
        2 * precision * recall / (precision + recall), 8
    ) if precision + recall else 0.0


datasets = {
    "UK validation": (
        RUN_DIR / "val_predictions_best.jsonl", "uk"
    ),
    "RU validation": (
        RUN_DIR / "val_predictions_best.jsonl", "ru"
    ),
    "UK original diagnostic": (
        RUN_DIR / "uk_testlike_original_predictions.jsonl", "uk"
    ),
    "UK testlike diagnostic": (
        RUN_DIR / "uk_testlike_predictions.jsonl", "uk"
    ),
}

results = []

for name, (path, source) in datasets.items():
    results.append({
        "dataset": name,
        "original_f1": evaluate_saved(path, normalize_original, source),
        "fixed_f1": evaluate_saved(path, normalize_fixed, source),
    })

display(pd.DataFrame(results))

,dataset,original_f1,fixed_f1
0,UK validation,0.866313,0.872238
1,RU validation,0.868048,0.867766
2,UK original diagnostic,0.855365,0.861885
3,UK testlike diagnostic,0.765785,0.771513


In [17]:
labeling = pd.read_csv(DATA_DIR / "labeling_sample.csv")

labeling_frame = (
    labeling[["text", "loc_markers"]]
    .copy()
    .reset_index(drop=True)
)

labeling_records = prepare_frame(
    labeling_frame,
    "labeling",
    "diagnostic",
    keep_for_evaluation=True
)

labeling_predictions, _ = predict_loader(
    inference_loader(labeling_records),
    "Labeling sample"
)

print(
    "Labeling sample F1:",
    comp_metric(
        [r["gold_locations"] for r in labeling_records],
        [
            labeling_predictions[r["text_id"]]["locations"]
            for r in labeling_records
        ]
    )
)

errors_table = []

for record in labeling_records:
    predicted = labeling_predictions[record["text_id"]]["locations"]
    gold = record["gold_locations"]

    errors_table.append({
        "text_id": record["text_id"],
        "text": record["text"],
        "gold": gold,
        "predicted": predicted,
        **location_errors(gold, predicted)
    })

display(pd.DataFrame(errors_table))

Prepare labeling/diagnostic:   0%|          | 0/1 [00:00<?, ?it/s]

Labeling sample:   0%|          | 0/1 [00:00<?, ?it/s]

Labeling sample F1: 0.48484848


,text_id,text,gold,predicted,false_positives,false_negatives
0,labeling_0,-261.310,[],[],[],[]
1,labeling_1,"Проспект Бандери стоїть, як і пів міста… довго...",[Проспект Бандери],"[Бандери, \n]","[, бандери]",[проспект бандери]
2,labeling_2,☢️Радіаційний фон (мкР/год) по Київщині станом...,"[Київщині, Чорнобиль, Київ, Тетерів, Фастів, Б...","[Київщині, \n\nЧорнобиль, \n, Київ, \nФ, \nМир...",[],"[біла церква, тетерів, фастів]"
3,labeling_3,"Друзі. Останні дві ночі, і ще декілька наступн...",[проспекті Івасюка],[проспекті Івасюка],[],[]
4,labeling_4,Сьогодні з Києва до Хелма вперше вирушив новий...,"[Києва, Хелма]","[Києва, Хелма, Укрзалізниці🚊\n\n]",[укрзалізниці🚊],[]
5,labeling_5,Подовження роботи закладів харчування\n\nВладо...,[Києву],"[Києву, Україні]",[україні],[]
6,labeling_6,"Нарешті офіційні новини, які можна розповідати...",[],"[\n, України]","[, україни]",[]
7,labeling_7,ДТП біля заїзду в ТЦ Аркадія\n\n📍Індустріальни...,[ТЦ Аркадія],"[ТЦ Аркадія\n\n📍, 📍, дустрі]","[дустрі, тц аркадія 📍, 📍]",[тц аркадія]
8,labeling_8,“Київський метрополітен” отримав ще 6 вагонів ...,[],[\n\n🇵🇱],[🇵🇱],[]
9,labeling_9,В результате массированной атаки врага ночью в...,"[Фастовском, Бучанском]","[Фастовском, Бучанском, \n]",[],[]


In [34]:
for i in range(0,26):
    record = labeling_records[i]
    prediction = labeling_predictions[record["text_id"]]["locations"]

    print("ID:", record["text_id"])
    print("Gold:", repr(record["gold_locations"]))
    print("Prediction:", repr(prediction))
    print("Errors:", location_errors(record["gold_locations"], prediction))
    print()

ID: labeling_0
Gold: []
Prediction: []
Errors: {'false_positives': [], 'false_negatives': []}

ID: labeling_1
Gold: ['Проспект Бандери']
Prediction: ['Бандери', '\n']
Errors: {'false_positives': ['', 'бандери'], 'false_negatives': ['проспект бандери']}

ID: labeling_2
Gold: ['Київщині', 'Чорнобиль', 'Київ', 'Тетерів', 'Фастів', 'Біла Церква', 'Миронівка', 'Бориспіль', 'Яготин', 'Баришівка']
Prediction: ['Київщині', '\n\nЧорнобиль', '\n', 'Київ', '\nФ', '\nМиронівка', '\nБориспіль', '\nЯготин', '\n', 'Баришівка']
Errors: {'false_positives': [''], 'false_negatives': ['біла церква', 'тетерів', 'фастів']}

ID: labeling_3
Gold: ['проспекті Івасюка']
Prediction: ['проспекті Івасюка']
Errors: {'false_positives': [], 'false_negatives': []}

ID: labeling_4
Gold: ['Києва', 'Хелма']
Prediction: ['Києва', 'Хелма', 'Укрзалізниці🚊\n\n']
Errors: {'false_positives': ['укрзалізниці🚊'], 'false_negatives': []}

ID: labeling_5
Gold: ['Києву']
Prediction: ['Києву', 'Україні']
Errors: {'false_positives': ['

In [18]:
def clean_locations(locations):
    cleaned = []

    for location in locations:
        location = location.strip()

        if location:
            cleaned.append(location)

    return cleaned


gold = [r["gold_locations"] for r in labeling_records]

pred_original = [
    labeling_predictions[r["text_id"]]["locations"]
    for r in labeling_records
]

pred_cleaned = [
    clean_locations(locations)
    for locations in pred_original
]

print("Original F1:", comp_metric(gold, pred_original))
print("Cleaned F1:", comp_metric(gold, pred_cleaned))

Original F1: 0.48484848
Cleaned F1: 0.53333333


In [21]:
import unicodedata

def clean_locations_v2(locations):
    cleaned = []

    for location in locations:
        # Remove symbols such as emoji, while keeping letters,
        # digits, punctuation, and whitespace.
        location = "".join(
            ch for ch in location
            if not unicodedata.category(ch).startswith("So")
        )

        location = location.strip()

        if location:
            cleaned.append(location)

    return cleaned


pred_v2 = [
    clean_locations_v2(locations)
    for locations in pred_original
]

print("Original F1:", comp_metric(gold, pred_original))
print("Strip F1:", comp_metric(gold, pred_cleaned))
print("Clean v2 F1:", comp_metric(gold, pred_v2))

Original F1: 0.48484848
Strip F1: 0.53333333
Clean v2 F1: 0.63414634


In [26]:
def clean_locations_v3(locations):
    cleaned = clean_locations_v2(locations)

    return [
        loc for loc in cleaned
        if any(ch.isalnum() for ch in loc)
    ]

In [27]:
def evaluate_combined_cleaning(path, source=None):
    rows = []

    with open(path, encoding="utf-8") as file:
        for line in file:
            row = json.loads(line)

            if source is None or row["source"] == source:
                rows.append(row)

    gold = [row["gold_locations"] for row in rows]
    original = [row["locations"] for row in rows]

    return {
        "original_f1": comp_metric(gold, original),
        "strip_f1": comp_metric(
            gold,
            [clean_locations(x) for x in original]
        ),
        "clean_v2_f1": comp_metric(
            gold,
            [clean_locations_v2(x) for x in original]
        ),
        "combined_v3_f1": comp_metric(
            gold,
            [clean_locations_v3(x) for x in original]
        )
    }


comparison = []

for name, (path, source) in datasets.items():
    comparison.append({
        "dataset": name,
        **evaluate_combined_cleaning(path, source)
    })

display(pd.DataFrame(comparison))

,dataset,original_f1,strip_f1,clean_v2_f1,combined_v3_f1
0,UK validation,0.866313,0.866313,0.865994,0.866040
1,RU validation,0.868048,0.868048,0.868048,0.868104
2,UK original diagnostic,0.855365,0.855365,0.854772,0.854772
3,UK testlike diagnostic,0.765785,0.778907,0.778785,0.778785


In [28]:
pred_v3 = [
    clean_locations_v3(locations)
    for locations in pred_original
]

print("Original:", comp_metric(gold, pred_original))
print("Strip:", comp_metric(gold, pred_cleaned))
print("Clean v2:", comp_metric(gold, pred_v2))
print("Combined v3:", comp_metric(gold, pred_v3))

Original: 0.48484848
Strip: 0.53333333
Clean v2: 0.63414634
Combined v3: 0.63414634


In [23]:
import ast
import json

def parse_locations(value):
    if isinstance(value, list):
        return value
    if isinstance(value, str):
        parsed = json.loads(value)
        if isinstance(parsed, list):
            return parsed
    raise ValueError(f"Unexpected locations format: {value!r}")


baseline_submission = pd.read_csv(SUBMISSION_PATH)

submission_strip = baseline_submission.copy()
submission_v2 = baseline_submission.copy()

submission_strip["locations"] = (
    baseline_submission["locations"]
    .map(parse_locations)
    .map(clean_locations)
    .map(json.dumps)
)

submission_v2["locations"] = (
    baseline_submission["locations"]
    .map(parse_locations)
    .map(clean_locations_v2)
    .map(json.dumps)
)

strip_path = RUN_DIR / "submission_strip.csv"
v2_path = RUN_DIR / "submission_clean_v2.csv"

submission_strip.to_csv(strip_path, index=False)
submission_v2.to_csv(v2_path, index=False)

print("Original:", SUBMISSION_PATH)
print("Strip:", strip_path)
print("Clean v2:", v2_path)

Original: /content/outputs/bilstm_20261110T192416Z_a87f4fa7/submission.csv
Strip: /content/outputs/bilstm_20261110T192416Z_a87f4fa7/submission_strip.csv
Clean v2: /content/outputs/bilstm_20261110T192416Z_a87f4fa7/submission_clean_v2.csv


In [24]:
assert list(baseline_submission.columns) == ["text_id", "locations"]

assert baseline_submission["text_id"].equals(
    submission_strip["text_id"]
)
assert baseline_submission["text_id"].equals(
    submission_v2["text_id"]
)

for frame in [baseline_submission, submission_strip, submission_v2]:
    assert frame["text_id"].is_unique
    assert frame["locations"].notna().all()
    assert frame["locations"].map(
        lambda x: isinstance(json.loads(x), list)
    ).all()

print(
    "Changed rows (strip):",
    (baseline_submission["locations"] != submission_strip["locations"]).sum()
)

print(
    "Changed rows (v2):",
    (baseline_submission["locations"] != submission_v2["locations"]).sum()
)

print("Validation passed!")

Changed rows (strip): 368
Changed rows (v2): 368
Validation passed!


In [29]:
submission_v3 = baseline_submission.copy()

submission_v3["locations"] = (
    baseline_submission["locations"]
    .map(parse_locations)
    .map(clean_locations_v3)
    .map(lambda locations: json.dumps(locations, ensure_ascii=False))
)

v3_path = RUN_DIR / "submission_clean_v3.csv"

submission_v3.to_csv(v3_path, index=False)

assert submission_v3["text_id"].equals(
    baseline_submission["text_id"]
)
assert submission_v3["text_id"].is_unique
assert submission_v3["locations"].map(
    lambda value: isinstance(json.loads(value), list)
).all()

print("Saved:", v3_path)

print(
    "Changed rows:",
    (
        baseline_submission["locations"]
        != submission_v3["locations"]
    ).sum()
)

Saved: /content/outputs/bilstm_20261110T192416Z_a87f4fa7/submission_clean_v3.csv
Changed rows: 190
